# Ingeniería de características Spotify

* Objetivo

El objetivo de este notebook es realizar la ingeniería de características sobre el dataset previamente limpiado y winsorizado.

En esta etapa se realizan principalmente las siguientes transformaciones:

* Carga del dataset `spotify_winsorizado.csv`.
* Conversión de la duración de las canciones desde milisegundos a minutos.
* Creación de una variable categórica para representar la duración de las canciones.
* Evaluación exploratoria de una alternativa basada en cuantiles.
* Eliminación de la variable original `duration_ms`.
* Transformación de popularity en una variable categórica ordinal denominada popularity_category.
* Exportación del dataset resultante como `spotify_engineering.csv`.

Esta etapa prepara los datos para el posterior preprocesamiento y modelamiento mediante Machine Learning.

---

# Importación de librerías

Se comienza importando las librerías necesarias:

In [ ]:
import pandas as pd
import os
import sys

"""
En caso de que Python no encuentre en la ruta los otros directorios,
ejecutar esta configuración
"""

sys.path.append(os.path.abspath(".."))

"""
Al ser utilidades locales del repositorio
se deben realizar la importación luego
del sys.path()
"""

---

# Tipos de variables

El dataset contiene diferentes tipos de variables que pueden clasificarse conceptualmente según su función dentro del análisis.

Identificación/texto: 
* *`artists`*
* *`album_name`*
* *`track_name`*
* *`track_genre`*

Estas variables contienen información textual relacionada con la canción, artista, álbum y género musical.

Debido a su naturaleza categórica y a su elevada cardinalidad, posteriormente requerirán técnicas específicas de codificación para poder ser utilizadas por los modelos de Machine Learning.

Variable objetivo: 
* *`popularity`*

Esta variable será posteriormente transformada en categorías ordinales para definir el objetivo del modelo predictivo.

Audio: 
* *`danceability`*,
* *`energy`*
* *`loudness`*
*  *`speechiness`*
* *`acousticness`*
* *`instrumentalness`* 
* *`liveness`*
* *`valence`*
* *`tempo`*

Estas variables representan diferentes características acústicas de las canciones.

Otras: 
* *`duration_ms`*
* *`explicit`*
* *`key`*
* *`mode`*

Estas variables contienen información adicional sobre cada canción.


---

# Carga del dataset

In [ ]:
data = pd.read_csv("../dataset/spotify_winsorizado.csv")

df = data.copy()

---

# Inspección de `duration_ms`

Se inspecciona inicialmente la variable:

In [38]:
df['duration_ms']

0         230666
1         149610
2         210826
3         201933
4         198853
           ...  
113417    384999
113418    385000
113419    271466
113420    283893
113421    241826
Name: duration_ms, Length: 113422, dtype: int64

La variable `duration_ms` representa la duración de cada canción en milisegundos.

Aunque esta unidad resulta adecuada para almacenar valores numéricos con precisión, no es especialmente intuitiva para realizar análisis exploratorios.

Por este motivo, se genera una nueva representación de la duración expresada en minutos.

---

# Conversión de duración de milisegundos a minutos

Se crea la nueva variable `duration_min`:

In [39]:
df["duration_min"] = df["duration_ms"] / 60000

df['duration_min']

0         3.844433
1         2.493500
2         3.513767
3         3.365550
4         3.314217
            ...   
113417    6.416650
113418    6.416667
113419    4.524433
113420    4.731550
113421    4.030433
Name: duration_min, Length: 113422, dtype: float64

## Ventaja de la transformación

Trabajar con minutos facilita:

* La interpretación de los valores.
* El análisis estadístico.
* La generación de visualizaciones.
* La construcción posterior de categorías de duración.

---

# Determinación de bins mediante Sturges

In [43]:
# Esta variable se utilizará solo para fines exploratorios
# No se incluirá en el modelo de ML

df['duration_category'] = pd.cut(
    df["duration_min"],
    bins=[0, 2.5, 4, 6, float("inf")],
    labels=["Corta", "Media", "Larga", "Muy larga"]
)

df.describe(include='category')

,duration_category
count,113422
unique,4
top,Media
freq,59307


---

ESTA OPCIÓN DE TRABAJO SE DEBE CONSULTAR 1RO CON EL PROFESOR PARA SU APROBACIÓN

In [45]:
# # Crear 5 categorías con la misma cantidad de canciones
# df['duration_cat_quantil'] = pd.qcut(
#     df['duration_min'], 
#     q=5,  # Puedes poner 4, 5, 6 o 10 según necesites
#     labels=['Muy corta', 'Corta', 'Media', 'Larga', 'Muy larga']
# )

# # Verifica que cada categoría tenga ~20% de los datos
# print(df['duration_cat_quantil'].value_counts(normalize=True))

---

# Eliminación columna `duration_ms`

Al haber creado la columna `duration_min` es redundante tener la columna `duration_ms` por lo que se eliminirá del dataset.

In [46]:
df.drop(columns=["duration_ms"], inplace=True)

---

# Creación de la variable ordinal `popularity_category`

In [49]:
df["popularity_category"] = pd.cut(
    df["popularity"],
    bins=[0, 20, 40, 60, 80, 100],
    labels=["Muy baja", "Baja", "Media", "Alta", "Muy alta"],
    include_lowest=True
)

---

# Exportación del dataset tratado

Una vez finalizado el proceso de ingeniería de datos, el dataset resultante se exporta en formato CSV:

In [ ]:
df.to_csv("../dataset/spotify_engineering.csv", index=False)